# Section 7 · Observability 5 — AgentCore Observability (CloudWatch) from Anywhere (Groq edition)

> **Groq edition.** The agent's model runs on Groq; the traces still go to CloudWatch, so you need AWS credentials (but no Bedrock model access) and a `GROQ_API_KEY` in your environment or a `.env` file.

Agents on AgentCore Runtime get observability automatically. Agents running elsewhere (your laptop, Lambda, EC2, ECS) can send the same traces to **CloudWatch GenAI Observability** with the AWS Distro for OpenTelemetry (ADOT).

### Prerequisites
1. AWS credentials configured
2. **CloudWatch Transaction Search** enabled (one time per account/Region): CloudWatch console → *Application Signals* → *Transaction Search* → *Enable*. See [the docs](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html).
3. A CloudWatch log group and stream (created below)

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present; the agent script inherits it

# The model runs on Groq; CloudWatch still needs an AWS Region
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)
print("GROQ_API_KEY:", "set" if os.getenv("GROQ_API_KEY") else "MISSING")

### Create a CloudWatch log group and log stream

In [ ]:
import boto3

LOG_GROUP = "agents/observability-demo"
LOG_STREAM = "default"

logs_client = boto3.client("logs", region_name=REGION)

try:
    logs_client.create_log_group(logGroupName=LOG_GROUP)
    print(f"Created log group: {LOG_GROUP}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"Log group already exists: {LOG_GROUP}")

try:
    logs_client.create_log_stream(logGroupName=LOG_GROUP, logStreamName=LOG_STREAM)
    print(f"Created log stream: {LOG_STREAM}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"Log stream already exists: {LOG_STREAM}")

### The agent as a standalone script

ADOT instruments a whole Python *process* via the `opentelemetry-instrument` command, so the agent runs as a script, not inside a notebook cell. This cell writes `agentcore_agent_groq.py` (the same file that is in this folder).

In [ ]:
%%writefile agentcore_agent_groq.py
"""Customer account agent used by the CloudWatch / ADOT tracing notebook.

The notebook runs it with `opentelemetry-instrument python agentcore_agent_groq.py`
so the AWS Distro for OpenTelemetry can capture traces at the process level.

Groq edition of agentcore_agent.py: only the model provider changes (needs GROQ_API_KEY).
"""

import os

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel

load_dotenv()

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")


@tool
def lookup_account(account_id: str) -> str:
    """Look up account details by account ID."""
    # Simulated account data
    accounts = {
        "ACC-001": {"name": "Acme Corp", "tier": "Enterprise", "balance": 142500.00},
        "ACC-002": {"name": "Startup Inc", "tier": "Growth", "balance": 28750.00},
    }
    account = accounts.get(account_id)
    if account:
        return f"Account {account_id}: {account['name']}, Tier: {account['tier']}, Balance: ${account['balance']:,.2f}"
    return f"Account {account_id} not found."


@tool
def calculate_discount(amount: float, tier: str) -> str:
    """Calculate discount based on customer tier."""
    rates = {"Enterprise": 0.15, "Growth": 0.10, "Starter": 0.05}
    rate = rates.get(tier, 0.0)
    discount = amount * rate
    return f"{tier} tier discount: {rate * 100:.0f}% off ${amount:,.2f} = ${discount:,.2f} savings"


model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer is returned.
    params={"extra_body": {"include_reasoning": False}},
)

agent = Agent(
    model=model,
    system_prompt="You are a customer account assistant. Use the available tools to help with account inquiries.",
    tools=[lookup_account, calculate_discount],
)

response = agent("Look up account ACC-001 and calculate what discount they would get on a $50,000 purchase.")
print(response)

### ADOT configuration

These environment variables tell ADOT where to send traces and how to label your agent in CloudWatch.

In [ ]:
%%writefile .env.agentcore
OTEL_PYTHON_DISTRO=aws_distro
OTEL_PYTHON_CONFIGURATOR=aws_configurator
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_TRACES_EXPORTER=otlp
OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group=agents/observability-demo,x-aws-log-stream=default,x-aws-metric-namespace=bedrock-agentcore
OTEL_RESOURCE_ATTRIBUTES=service.name=s07-account-agent,aws.log.group.names=agents/observability-demo
AGENT_OBSERVABILITY_ENABLED=true

### Run the agent with ADOT instrumentation

In [ ]:
import shutil
import subprocess
import sys

from dotenv import dotenv_values

env = {**os.environ, **dotenv_values(".env.agentcore"), "AWS_REGION": REGION, "AWS_DEFAULT_REGION": REGION}

# Use the opentelemetry-instrument launcher from the same environment as this notebook
launcher = os.path.join(os.path.dirname(sys.executable), "opentelemetry-instrument")
if not os.path.exists(launcher):
    launcher = shutil.which("opentelemetry-instrument")

result = subprocess.run([launcher, sys.executable, "agentcore_agent_groq.py"], env=env, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

### View the traces

Open the [CloudWatch GenAI Observability dashboard](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability) (allow a minute or two). Find **s07-account-agent** and drill into a trace: the agent invocation, the model calls with token counts, and the `lookup_account` and `calculate_discount` tool calls with their inputs and outputs.

It is the same trace data Jaeger, Langfuse and LangSmith show — viewed through CloudWatch.

### Clean up

In [ ]:
logs_client.delete_log_group(logGroupName=LOG_GROUP)
print(f"Deleted log group: {LOG_GROUP}")